# The Topic Relevance Score: Cosine of the Firm Summary and the Policy Summary

| Item | Description |
|---|---|
| Goal | One topic relevance score per firm. It is the cosine similarity between the embedding of the firm's summary of its 10-K Item 1 sections (T3.3) and the embedding of the policy summary of legislator statements on consumer data privacy legislation (T3.2). The cosine of each yearly summary gives, in addition, one value per filing. |
| Input | `firm_summary_embeddings.npy`, `filing_summary_embeddings.npy`, their key files, and the two summary tables from T3.3 in the T3.3 Processed folder. `policy_summary_embedding.npy` and `policy_summary_text.csv` from T3.2.5; `filings.parquet` from T3.1.1. The CRSP-Compustat link file from D3 (licensed, on the scratch root only) for each firm's NAICS code. |
| Output | In the T3.4 Processed folder: `firm_relevance.parquet` (one row per firm and sample) and `filing_cosines.parquet` (one row per filing). Under `R - Results/tables/...`: `table_firm_relevance_scores.csv`, `table_filing_cosines.csv`, `table_score_quantiles.csv`, and `table_example_meta.csv`. |
| Variables | • `score`: the cosine of the firm summary with the policy summary, the main measure<br>• `score_yearly_mean`: the mean of the firm's yearly cosines over its filings in the sample<br>• `sector`: the firm's Compustat NAICS code cut to two digits, with 31 to 33 merged into manufacturing, 44 to 45 into retail trade, and 48 to 49 into transportation and warehousing |
| Sample | Both samples of T3.1.1, with the all-years sample as the main one: every fiscal year a firm filed, 1999 to 2025, 1,058 firms. Prompt version `item1-mapreduce-1.3` of the `summary_lines_of_business` family. |
| Run | Run 2026-10-31. The scores equal those of the October 2026 run, which the proposal reports. |


## Step 1. Import packages


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


## Step 2. Define file locations and settings


In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure/T3.4 - Topic Relevance - Score and Figures")
NOTEBOOK_NAME = "T3.4.1 - Score - Cosine of the Firm Summary and the Policy Summary"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
T32_PROCESSED = SCRATCH_FOLDER / "T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure/T3.2 - Topic Side - Policy Summary of the Legislator Statements" / "Processed"
T33_PROCESSED = SCRATCH_FOLDER / "T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure/T3.3 - Firm Side - Summaries of the 10-K Item 1 Filings" / "Processed"
FILINGS_FILE = SCRATCH_FOLDER / "T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure/T3.1 - Sample - Firms and 10-K Item 1 Filings" / "Processed" / "filings.parquet"
LINK_FILE = SCRATCH_FOLDER / "D - Data Collection and Processing/D3 - SEC Form 10-K Corporate Filings" / "Raw" / "firm_id" / "crsp_compustat_link.dta"  # licensed WRDS file, never in the repository

SAMPLES = ["all_years", "three_years"]  # the all-years sample is the main one
FAMILY = "summary_lines_of_business"
VERSION = "item1-mapreduce-1.3"
EXAMPLE_FIRM = "META PLATFORMS INC"
NAICS_SECTORS = {  # two-digit code -> sector name; codes that share a sector map to the same name
    "11": "Agriculture, forestry, fishing and hunting", "21": "Mining, quarrying, and oil and gas extraction", "22": "Utilities", "23": "Construction",
    "31": "Manufacturing", "32": "Manufacturing", "33": "Manufacturing", "42": "Wholesale trade", "44": "Retail trade", "45": "Retail trade",
    "48": "Transportation and warehousing", "49": "Transportation and warehousing", "51": "Information", "52": "Finance and insurance",
    "53": "Real estate and rental and leasing", "54": "Professional, scientific, and technical services", "55": "Management of companies and enterprises",
    "56": "Administrative and support and waste management", "61": "Educational services", "62": "Health care and social assistance",
    "71": "Arts, entertainment, and recreation", "72": "Accommodation and food services", "81": "Other services", "92": "Public administration",
    "99": "Unclassified",
}
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure" / ".env"
for folder in (PROCESSED_FOLDER, TABLE_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)


## Step 3. The topic side: the policy summary vector

One unit vector, the embedding of the policy summary that T3.2.3 wrote from the 374 statements. The text is printed so the reader sees what every firm is measured against.


In [3]:
policy_vector = np.load(T32_PROCESSED / "policy_summary_embedding.npy").reshape(-1).astype(np.float32)
policy = pd.read_csv(T32_PROCESSED / "policy_summary_text.csv").iloc[0]
assert policy_vector.shape == (1024,) and abs(np.linalg.norm(policy_vector) - 1) < 1e-4
print(f"POLICY SUMMARY VECTOR = embedding of this text ({policy['n_words']} words, {policy['n_tokens']} tokens):\n")
print(policy["policy_summary"])


POLICY SUMMARY VECTOR = embedding of this text (221 words, 274 tokens):

The legislation imposes rigorous obligations on tech giants, broadband providers, healthcare organizations, drone operators, and financial institutions to secure consumer data and prevent its unauthorized sale or misuse. It grants individuals specific rights, including the ability to opt in or out of data collection, demand clear and conspicuous privacy policies, control their digital communications, and establish data "firewalls." Companies are required to implement "privacy by design," provide breach notifications within 72 hours, and ensure that sensitive information such as health records, social security numbers, and geolocation data is protected from discrimination and identity theft.

The regulatory scope targets a wide array of data practices, ranging from the deceptive monitoring of online activities and the unauthorized mapping of consumer movements to the commercialization of private data and the use of

## Step 4. The firm side: the yearly summaries and the firm summaries

T3.3.2 wrote one summary per sampled filing, T3.3.4 one firm summary per sample, and T3.3.6 embedded them all. Only the chosen family and version are kept here.


In [4]:
filing_keys = pd.read_parquet(T33_PROCESSED / "filing_summary_embedding_keys.parquet")
filing_vectors = np.load(T33_PROCESSED / "filing_summary_embeddings.npy")
firm_keys = pd.read_parquet(T33_PROCESSED / "firm_summary_embedding_keys.parquet")
firm_vectors = np.load(T33_PROCESSED / "firm_summary_embeddings.npy")
assert len(filing_keys) == len(filing_vectors) and len(firm_keys) == len(firm_vectors)
keep = (filing_keys["family"].eq(FAMILY) & filing_keys["prompt_version"].eq(VERSION)).to_numpy()
filing_keys, filing_vectors = filing_keys.loc[keep].reset_index(drop=True), filing_vectors[keep]
keep = (firm_keys["set"].isin(SAMPLES) & firm_keys["family"].eq(FAMILY) & firm_keys["prompt_version"].eq(VERSION)).to_numpy()
firm_keys, firm_vectors = firm_keys.loc[keep].reset_index(drop=True), firm_vectors[keep]
assert np.allclose(np.linalg.norm(filing_vectors, axis=1), 1, atol=1e-4) and np.allclose(np.linalg.norm(firm_vectors, axis=1), 1, atol=1e-4)

filings = pd.read_parquet(FILINGS_FILE)
filing_table = filing_keys.merge(filings[["accessionNo", "gvkey", "conm", "fyear", "in_three_years", "in_all_years"]], on="accessionNo", how="left", validate="one_to_one")
assert filing_table["gvkey"].notna().all() and len(filing_table) == len(filings)
firm_names = filings.drop_duplicates("gvkey").set_index("gvkey")["conm"]
for sample in SAMPLES:
    assert set(firm_keys.loc[firm_keys["set"].eq(sample), "gvkey"]) == set(filings.loc[filings["in_" + sample].eq(1), "gvkey"]), sample
filing_texts = pd.read_parquet(T33_PROCESSED / "filing_summaries.parquet", columns=["accessionNo", "family", "prompt_version", "summary", "n_words"])
filing_texts = filing_texts.loc[filing_texts["family"].eq(FAMILY) & filing_texts["prompt_version"].eq(VERSION)].set_index("accessionNo")
firm_texts = pd.read_parquet(T33_PROCESSED / "firm_summaries.parquet", columns=["gvkey", "set", "family", "prompt_version", "summary", "n_words"])
firm_texts = firm_texts.loc[firm_texts["family"].eq(FAMILY) & firm_texts["prompt_version"].eq(VERSION)].set_index(["set", "gvkey"])
print(f"Yearly summaries: {len(filing_table):,} ({filing_table['gvkey'].nunique():,} firms); firm summaries: " + ", ".join(f"{sample} {int(firm_keys['set'].eq(sample).sum()):,}" for sample in SAMPLES))


Yearly summaries: 18,779 (1,058 firms); firm summaries: all_years 1,058, three_years 841


## Step 5. Compute the score

For firm $i$, let $f_i$ be the embedding of its firm summary and $e_{it}$ the embedding of its yearly summary for fiscal year $t$. $T_i$ is its fiscal years in the sample and $p$ the policy summary embedding. All embeddings have unit length, so the cosine is the dot product.

$$\text{score}_i = \cos(f_i, p), \qquad \text{score}^{\text{yearly}}_i = \frac{1}{|T_i|} \sum_{t \in T_i} \cos(e_{it}, p)$$

The first is the main measure; the second averages the per-filing cosines and is kept for comparison.


In [5]:
filing_table["cos_policy_summary"] = filing_vectors @ policy_vector
firm_keys["score"] = firm_vectors @ policy_vector
rows = []
for sample in SAMPLES:
    in_sample = filing_table.loc[filing_table["in_" + sample].eq(1)]
    yearly = in_sample.groupby("gvkey").agg(n_filings=("accessionNo", "size"), score_yearly_mean=("cos_policy_summary", "mean"))
    part = firm_keys.loc[firm_keys["set"].eq(sample), ["gvkey", "score"]].merge(yearly, left_on="gvkey", right_index=True, how="left", validate="one_to_one")
    rows.append(part.assign(sample=sample))
scores = pd.concat(rows, ignore_index=True)
scores["conm"] = scores["gvkey"].map(firm_names)
assert scores["score"].notna().all() and scores["n_filings"].notna().all()
print(f"Scores: {len(scores):,} firm-sample rows; " + "; ".join(f"{sample}: {int(scores['sample'].eq(sample).sum()):,} firms" for sample in SAMPLES))


Scores: 1,899 firm-sample rows; all_years: 1,058 firms; three_years: 841 firms


## Step 6. Attach each firm's NAICS sector

The firm's Compustat NAICS code from the CRSP-Compustat link file, cut to two digits. Every firm must have one code.


In [6]:
link = pd.read_stata(LINK_FILE, columns=["GVKEY", "naics"])
link["gvkey"] = link["GVKEY"].astype(str).str.zfill(6)
link["naics"] = link["naics"].astype(str).str.strip().replace({"": None, "nan": None})
link = link.dropna(subset=["naics"])
assert link.groupby("gvkey")["naics"].nunique().le(1).all(), "a gvkey with two NAICS codes"
naics = link.drop_duplicates("gvkey").set_index("gvkey")["naics"]
scores["naics"] = scores["gvkey"].map(naics)
assert scores["naics"].notna().all(), f"{int(scores['naics'].isna().sum())} firms without a NAICS code"
scores["naics2"] = scores["naics"].str[:2]
assert scores["naics2"].isin(NAICS_SECTORS).all(), sorted(set(scores["naics2"]) - set(NAICS_SECTORS))
scores["sector"] = scores["naics2"].map(NAICS_SECTORS)
scores = scores[["gvkey", "conm", "sample", "n_filings", "naics", "naics2", "sector", "score", "score_yearly_mean"]]
print(f"Sectors in the all-years sample: {scores.loc[scores['sample'].eq('all_years'), 'sector'].nunique()}")
display(scores.loc[scores["sample"].eq("all_years")].groupby("sector")["gvkey"].size().sort_values(ascending=False).rename("firms").to_frame().T)


Sectors in the all-years sample: 19


sector,Manufacturing,Finance and insurance,Information,Retail trade,Utilities,"Mining, quarrying, and oil and gas extraction",Real estate and rental and leasing,"Professional, scientific, and technical services",Transportation and warehousing,Wholesale trade,Accommodation and food services,Administrative and support and waste management,Health care and social assistance,Construction,"Arts, entertainment, and recreation",Unclassified,Other services,Educational services,"Agriculture, forestry, fishing and hunting"
firms,420,149,113,59,58,53,47,38,27,21,17,14,13,11,5,4,3,3,3


## Step 7. Worked example: Meta Platforms

The first and the last yearly summary and the firm summary of the all-years sample in full, then every year's cosine, their mean, and the firm-summary cosine, the score.


In [7]:
example_gvkey = firm_names.index[firm_names.eq(EXAMPLE_FIRM)]
assert len(example_gvkey) == 1
example_gvkey = example_gvkey[0]
example_filings = filing_table.loc[filing_table["gvkey"].eq(example_gvkey)].sort_values("fyear")
print(f"{EXAMPLE_FIRM} (gvkey {example_gvkey}), all-years sample: {len(example_filings)} yearly summaries, fiscal {example_filings['fyear'].min()} to {example_filings['fyear'].max()}")
for filing in [example_filings.iloc[0], example_filings.iloc[-1]]:
    text = filing_texts.loc[filing["accessionNo"]]
    print(f"\n--- yearly summary, fiscal {filing['fyear']} ({text['n_words']} words) ---\n{text['summary']}")
firm_text = firm_texts.loc[("all_years", example_gvkey)]
print(f"\n--- firm summary, all-years sample ({firm_text['n_words']} words) ---\n{firm_text['summary']}")
example = example_filings[["fyear", "cos_policy_summary"]].rename(columns={"cos_policy_summary": "cosine_with_policy_summary"}).set_index("fyear")
example.index = [f"yearly summary, fiscal {year}" for year in example.index]
example.loc["mean of the yearly cosines (score_yearly_mean)"] = example.mean()
example.loc["firm summary (score)"] = float(scores.loc[scores["gvkey"].eq(example_gvkey) & scores["sample"].eq("all_years"), "score"].iloc[0])
example.round(4).to_csv(TABLE_FOLDER / "table_example_meta.csv", index_label="text")
rank = int((scores.loc[scores["sample"].eq("all_years"), "score"] > example.loc["firm summary (score)", "cosine_with_policy_summary"]).sum()) + 1
print(f"\nMeta's score ranks {rank} of {int(scores['sample'].eq('all_years').sum()):,} firms in the all-years sample")
display(example.round(4))


META PLATFORMS INC (gvkey 170617), all-years sample: 14 yearly summaries, fiscal 2012 to 2025

--- yearly summary, fiscal 2012 (166 words) ---
META PLATFORMS INC. operates a social networking platform called Facebook that enables users to connect, share content, and interact with friends, developers, and marketers through features like News Feed, Timeline, and mobile apps. The company also provides Instagram and various mobile applications for photo-sharing, messaging, and micro-blogging, alongside a development platform offering APIs and tools for third-party developers to create and integrate social applications and websites. Additionally, it offers advertising services allowing marketers to reach, target, and engage with users via formats including sponsored stories, ads in the News Feed, and real-time bidding exchanges. The company's primary customers are individual users accessing the network for social interaction, developers utilizing the platform to build applications, and mark

,cosine_with_policy_summary
"yearly summary, fiscal 2012",0.5150
"yearly summary, fiscal 2013",0.5221
"yearly summary, fiscal 2014",0.5023
"yearly summary, fiscal 2015",0.5458
"yearly summary, fiscal 2016",0.5482
"yearly summary, fiscal 2017",0.5535
"yearly summary, fiscal 2018",0.5618
"yearly summary, fiscal 2019",0.5281
"yearly summary, fiscal 2020",0.5547
"yearly summary, fiscal 2021",0.5495


## Step 8. Describe the scores

Quantiles by sample, the Spearman correlation between the two aggregations, and the ten highest and lowest firms of the all-years sample.


In [8]:
quantiles = scores.groupby("sample")[["score", "score_yearly_mean"]].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).stack(level=0, future_stack=True).round(4)
quantiles.to_csv(TABLE_FOLDER / "table_score_quantiles.csv")
display(quantiles)
for sample in SAMPLES:
    part = scores.loc[scores["sample"].eq(sample)]
    print(f"{sample}: Spearman correlation of the score with the yearly mean {part['score'].corr(part['score_yearly_mean'], method='spearman'):.3f}")
main = scores.loc[scores["sample"].eq("all_years")].sort_values("score", ascending=False)
display(pd.concat([main.head(10)[["conm", "sector", "score"]].reset_index(drop=True), main.tail(10)[["conm", "sector", "score"]].reset_index(drop=True)], axis=1, keys=["highest", "lowest"]).round(3))


count    mean     std     min      5%     25%  \
sample                                                                          
all_years   score              1058.0  0.5121  0.0362  0.4034  0.4566  0.4862   
            score_yearly_mean  1058.0  0.5075  0.0357  0.3874  0.4526  0.4824   
three_years score               841.0  0.5143  0.0368  0.4010  0.4563  0.4881   
            score_yearly_mean   841.0  0.5078  0.0380  0.3870  0.4492  0.4808   

                                  50%     75%     95%     max  
sample                                                         
all_years   score              0.5106  0.5367  0.5730  0.6209  
            score_yearly_mean  0.5066  0.5319  0.5693  0.6167  
three_years score              0.5123  0.5399  0.5769  0.6224  
            score_yearly_mean  0.5065  0.5329  0.5742  0.6258

all_years: Spearman correlation of the score with the yearly mean 0.873
three_years: Spearman correlation of the score with the yearly mean 0.893


highest  \
                          conm   
0                     VISA INC   
1       PALANTIR TECHNOLOG INC   
2              GEN DIGITAL INC   
3      COMVERSE TECHNOLOGY INC   
4         NIELSEN HOLDINGS PLC   
5        DELL TECHNOLOGIES INC   
6    CHARTER ONE FINANCIAL INC   
7                 ALPHABET INC   
8   VERIZON COMMUNICATIONS INC   
9  MOTOROLA MOBILITY HLDGS INC   

                                                            \
                                             sector  score   
0                             Finance and insurance  0.621   
1                                       Information  0.617   
2                                       Information  0.616   
3                                       Information  0.613   
4  Professional, scientific, and technical services  0.613   
5                                     Manufacturing  0.611   
6                             Finance and insurance  0.610   
7                                       Information  0.609   
8                                       Information  0.607   
9                                     Manufacturing  0.606   

                        lowest                                                 \
                          conm                                         sector   
0         CLEVELAND-CLIFFS INC                                  Manufacturing   
1    MARTIN MARIETTA MATERIALS  Mining, quarrying, and oil and gas extraction   
2     PEPSI BOTTLING GROUP INC                                  Manufacturing   
3     BURLINGTON RESOURCES INC  Mining, quarrying, and oil and gas extraction   
4    PEP BOYS-MANNY MOE & JACK                                   Retail trade   
5      ANADARKO PETROLEUM CORP  Mining, quarrying, and oil and gas extraction   
6         FREEPORT-MCMORAN INC  Mining, quarrying, and oil and gas extraction   
7              VALARIS LIMITED  Mining, quarrying, and oil and gas extraction   
8   HUNTINGTON INGALLS IND INC                                  Manufacturing   
9  ALPHA NATURAL RESOURCES INC  Mining, quarrying, and oil and gas extraction   

          
   score  
0  0.436  
1  0.435  
2  0.429  
3  0.427  
4  0.425  
5  0.424  
6  0.421  
7  0.413  
8  0.406  
9  0.403

## Step 9. Save the scores and the filing cosines

One table per firm and sample with the sector, and one row per filing with its yearly cosine; both as data on the scratch root and as readable CSV under the results.


In [9]:
scores = scores.sort_values(["sample", "gvkey"]).reset_index(drop=True)
scores.to_parquet(PROCESSED_FOLDER / "firm_relevance.parquet", index=False)
scores.to_csv(TABLE_FOLDER / "table_firm_relevance_scores.csv", index=False, float_format="%.6f")
filing_cosines = filing_table[["accessionNo", "gvkey", "conm", "fyear", "in_three_years", "cos_policy_summary"]].sort_values(["gvkey", "fyear"]).reset_index(drop=True)
filing_cosines.to_parquet(PROCESSED_FOLDER / "filing_cosines.parquet", index=False)
filing_cosines.to_csv(TABLE_FOLDER / "table_filing_cosines.csv", index=False, float_format="%.6f")
print(f"Saved firm_relevance.parquet and table_firm_relevance_scores.csv ({len(scores):,} rows); filing_cosines.parquet and table_filing_cosines.csv ({len(filing_cosines):,} rows)")


Saved firm_relevance.parquet and table_firm_relevance_scores.csv (1,899 rows); filing_cosines.parquet and table_filing_cosines.csv (18,779 rows)
